In [ ]:
!pip install requests

In [ ]:
import requests
import json
from requests.auth import HTTPBasicAuth

BASE_URL = "http://hpappworx01:8088/ae/api/v1"
USERNAME = "NLR9894"
PASSWORD = "Technology@234Technology@234"
CLIENT = 3000

: 

In [ ]:
#need code to read the worflows from the table
# Get status of each workflow rather than getting status like the get_executions()


In [ ]:
#This will give the status and also output json depending on the status of the workflow
def get_executions():
    url = f"{BASE_URL}/{CLIENT}/executions"
    response = requests.get(
        url, auth=HTTPBasicAuth(USERNAME, PASSWORD),
        verify=False
    )
    return response

response = get_executions()
data = response.json()
print("Response Data:", json.dumps(data, indent=4))

: 

In [ ]:
def get_latest_execution_by_name(workflow_name):
    url = f"{BASE_URL}/{CLIENT}/executions"
    
    # Targeting the specific workflow and limiting to the most recent run
    params = {
        "name": workflow_name,
        "max_results": 10
    }
    
    response = requests.get(
        url, 
        auth=HTTPBasicAuth(USERNAME, PASSWORD),
        params=params,
        verify=False
    )
    
    if response.status_code == 200:
        return response.json().get("data", [])
    return []

: 

In [ ]:
#Get all jobs out of a successfully completed workflow
def get_successful_workflows():
    response = get_executions()
    data = response.json()

    filtered = []
    for job in data.get("data", []):
        if job.get("type") == "JOBP":       # job.get("status") == 1550 and
            filtered.append(job)
    return filtered

successful_workflows = get_successful_workflows()
print("filtered_workflows:", json.dumps(successful_workflows, indent=4))

In [ ]:

# step1: We will get the status and details using Automic API1 (this is just a placeholder name. I have the actual API detalis) for all the workflows saved in a table continously in a loop.
# step2: Call Automic API2 including the output from step1 (run_id and status) to get all the jobs/tasks from the workflow
    #step2.1: I want to make sure we only process the file moving jobs
# step3: Check the status of all jobs that we get from step2 of each workflow 
# step4: Depending on the status of each job we have different logic to apply
    # step4.1: if status = ended ok. get runtime and call OpenAI to parse logs. save that into a table
    # step4.2: if status = ended_not_ok. get runtime and call OpenAI to parse error logs.save that into a table 
    # step4.3: if status = aborted. extract abort details
    # step4.4: if status = waiting. get runtime
    # step4.5: if status = blocked. blocking conditions are identified
# step5: Save the result of each job into the final job processing table.
# step6: File integrity and business-level checks are done on processed files and parsed logs using technical and domain rule tables. eg:- file size, processed file count
# step7: Save pass/fail result to a table
# step8: Alerts are triggered based on the result of step6
# step9: Push results to monitoring dashboard


In [ ]:
def get_execution_details(run_id):
    url = f"{BASE_URL}/{CLIENT}/executions/{run_id}"
    response = requests.get(
        url, auth=HTTPBasicAuth(USERNAME, PASSWORD),
        verify=False
    )
    return response.json()

execution_details_map = {}
for workflow in successful_workflows:
    run_id = workflow.get("run_id")
    print("details for run_id:", run_id)
    details = get_execution_details(run_id)
    print(f"Details for run_id {run_id}:", json.dumps(details, indent=4))
    execution_details_map[run_id] = details

In [8]:
def get_children(run_id):
    url = f"{BASE_URL}/{CLIENT}/executions/{run_id}/children"
    response = requests.get(
        url, auth=HTTPBasicAuth(USERNAME, PASSWORD),
        verify=False
    )
    return response.json()

children_map = {}
for run_id in execution_details_map.keys():
    print("children for run_id:", run_id)
    children = get_children(run_id)
    print(f"Children for run_id {run_id}:", json.dumps(children, indent=4))
    children_map[run_id] = children

children for run_id: 86008267
Children for run_id 86008267: {
    "total": 9,
    "data": [
        {
            "name": "JOBS.WEEKLY_AVESIS_INVOICE_FILE_ARCHIVE_COPY",
            "type": "JOBS",
            "queue": "QUEUE.BATCH",
            "run_id": 86008291,
            "status": 1700,
            "status_text": "Waiting for predecessor",
            "activation_time": "2026-04-17T16:00:10Z",
            "agent": "HPAPPWPROD",
            "platform": "WINDOWS",
            "parent": 86008267,
            "reference_run_id": 0,
            "line_number": 10,
            "user": "DMACLACKLIN/HEALTHPARTNERS",
            "estimated_runtime": 10,
            "archive_key1": "SAS_JOBS",
            "archive_key2": "HP_EXECS_PKZIP",
            "title": "COPY_ARCHIVE_AVESIS_WILY_CLAIM:Copy files to the Landing Zone then archive",
            "alias": "JOBS.WEEKLY_AVESIS_INVOICE_FILE_ARCHIVE_COPY",
            "activator": 86008267,
            "activator_object_type": "JOBP",
        

In [9]:
def get_report_types(run_id):
    url = f"{BASE_URL}/{CLIENT}/executions/{run_id}/reports"
    response = requests.get(
        url, auth=HTTPBasicAuth(USERNAME, PASSWORD),
        verify=False
    )
    return response.json()

# print("report types for run_id:", get_report_types(run_id=85961351))

In [10]:
def get_report_content(run_id, report_type):
    url = f"{BASE_URL}/{CLIENT}/executions/{run_id}/reports/{report_type}"
    response = requests.get(
        url, auth=HTTPBasicAuth(USERNAME, PASSWORD),
        verify=False
    )
    data = response.json()

    contents = []
    for item in data.get("data", []):
        content = item.get("content")
        if content:
            contents.append(content)

    return "\n".join(contents)

In [11]:
reports_map = {}

for run_id in execution_details_map.keys():
    print("reports for run_id:", run_id)
    report_info = get_report_types(run_id)
    reports_map[run_id] = {}

    for report in report_info:
        report_type = report.get("type")
        print(f"report type: {report_type}")
        content = get_report_content(run_id, report_type)
        print(f"content for report type {report_type}:", content)
        reports_map[run_id][report_type] = content

reports for run_id: 86008267
report type: ACT
content for report type ACT: 2026-04-17 16:00:03 - U00020206 Variable '&RUN_ID' was stored with value '1229770789975781377'.
2026-04-17 16:00:05 - U00007000 'JOBS.SFTP_FTPTODAY_AVESGEN_WEEKLY_INVOICE_TEXT_DOWNLOAD' activated with RunID '0086008272'.
2026-04-17 16:00:05 - U00007000 'JOBS.SFTP_FTPTODAY_AVESGEN_WEEKLY_INVOICE_EXCEL_DOWNLOAD' activated with RunID '0086007599'.
2026-04-17 16:00:06 - U00007000 'JOBS.SFTP_FTPTODAY_AVESGEN_WEEKLY_INVOICE_PDF_DOWNLOAD' activated with RunID '0086006437'.
2026-04-17 16:00:07 - U00007000 'JOBS.SFTP_FTPTODAY_AVESGEN_WEEKLY_NJ_INVOICE_EXCEL_DOWNLOAD' activated with RunID '0086003808'.
2026-04-17 16:00:07 - U00007000 'JOBS.SFTP_FTPTODAY_AVESGEN_WEEKLY_NJ_INVOICE_PDF_DOWNLOAD' activated with RunID '0086006439'.
2026-04-17 16:00:09 - U00007000 'JOBS.COPY_AVESIS_INVOICE_HPAPPWORXTS' activated with RunID '0086006440'.
2026-04-17 16:00:10 - U00007000 'JOBS.WEEKLY_AVESIS_INVOICE_FILE_RENAME_COPY' activated with

In [12]:
children_run_ids_map = {}

for run_id, children_info in children_map.items():
    child_run_ids = []
    for job in children_info.get("data", []):
        child_run_id = job.get("run_id")
        if child_run_id:
            child_run_ids.append(child_run_id)
    children_run_ids_map[run_id] = child_run_ids

print(children_run_ids_map)

{86008267: [86008291, 86008290, 86008289, 86006440, 86006439, 86006437, 86003808, 86007599, 86008272], 86007387: [86007399, 86006362, 86008082, 86008078], 86007181: [86003683, 86005923, 86007194, 86005919, 86005916, 86006302, 86005909, 86003674], 86007167: [86007181, 86007179, 86006286, 86005898], 86005892: [86005904, 86003662, 86005902], 86004584: [86005463, 86005462]}


In [13]:
child_reports_map = {}

for parent_run_id, child_run_ids in children_run_ids_map.items():
    print(f"parent run_id: {parent_run_id}")
    child_reports_map[parent_run_id] = {}

    for child_run_id in child_run_ids:
        print(f"\n child run_id: {child_run_id}")
        report_info = get_report_types(child_run_id)
        print("REPORT TYPES:", report_info)
        child_reports_map[parent_run_id][child_run_id] = {}

        for report in report_info:
            report_type = report.get("type")
            print(f"report type: {report_type}")
            content = get_report_content(child_run_id, report_type)
            print(content)
            child_reports_map[parent_run_id][child_run_id][report_type] = content

parent run_id: 86008267

 child run_id: 86008291
REPORT TYPES: []

 child run_id: 86008290
REPORT TYPES: []

 child run_id: 86008289
REPORT TYPES: [{'type': 'ACT', 'is_db': True}]
report type: ACT
2026-04-17 16:00:10 - U00020206 Variable '&RUN_ID' was stored with value '1229770789975781377'.


 child run_id: 86006440
REPORT TYPES: [{'type': 'ACT', 'is_db': True}]
report type: ACT
2026-04-17 16:00:09 - U00020206 Variable '&RUN_ID' was stored with value '1229770789975781377'.


 child run_id: 86006439
REPORT TYPES: []

 child run_id: 86006437
REPORT TYPES: []

 child run_id: 86003808
REPORT TYPES: []

 child run_id: 86007599
REPORT TYPES: []

 child run_id: 86008272
REPORT TYPES: []
parent run_id: 86007387

 child run_id: 86007399
REPORT TYPES: [{'type': 'ACT', 'is_db': True}]
report type: ACT
2026-04-17 15:00:05 - U00020206 Variable '&RUN_ID' was stored with value '1229770789975781377'.


 child run_id: 86006362
REPORT TYPES: []

 child run_id: 86008082
REPORT TYPES: []

 child run_id: 

AttributeError: 'list' object has no attribute 'get'

In [14]:
save_data = []

for workflow in successful_workflows:
    workflow_id = workflow.get("run_id")
    print(f"\nProcessing workflow run_id: {workflow_id}")
    workflow_data = {
        "run_id": workflow_id,
        "name": workflow.get("name"),
        "details": execution_details_map.get(workflow_id),
        "reports": reports_map.get(workflow_id),
        "children": []
    }
    
    children_list = children_map.get(workflow_id, {}).get("data", [])
    
    print("child_run_ids:", child_run_ids)
    print("children_list:", len(children_list))
    
    print("sample children_list:")
    # for c in children_list[:3]:
    #     print(c)
        
    # children_lookup = {}
    # for child in children_list:
    #     key = str(child.get("run_id")).strip()
    #     children_lookup[key] = child
        
    # print("lookup keys first 5:", list(children_lookup.keys())[:5])
    
    for child in children_list:
        child_run_id = child.get("run_id")
        print("looking for children:", child_run_id, type(child_run_id))
        # # child_info = children_lookup.get(str(child_run_id), {})
        # lookup_key = str(child_run_id).strip()
        # print("lookup key used:", lookup_key)
        # child_info = children_lookup.get(lookup_key)
        # if not child_info:
        #     print(f"Warning: No child info found for run_id {child_run_id} in workflow {workflow_id}")
        # else:
        #     print("found:", child_info.get("name"))
            
        child_data = {
            "run_id": child_run_id,
            "name": child.get("name"),
            "details": child,
            "status": child.get("status"),
            "type": child.get("type"),
            "reports": child_reports_map.get(workflow_id, {}).get(child_run_id, {}),
        }
        workflow_data["children"].append(child_data)
        
    save_data.append(workflow_data)


Processing workflow run_id: 86008267
child_run_ids: [86005463, 86005462]
children_list: 9
sample children_list:
looking for children: 86008291 <class 'int'>
looking for children: 86008290 <class 'int'>
looking for children: 86008289 <class 'int'>
looking for children: 86006440 <class 'int'>
looking for children: 86006439 <class 'int'>
looking for children: 86006437 <class 'int'>
looking for children: 86003808 <class 'int'>
looking for children: 86007599 <class 'int'>
looking for children: 86008272 <class 'int'>

Processing workflow run_id: 86007387
child_run_ids: [86005463, 86005462]
children_list: 4
sample children_list:
looking for children: 86007399 <class 'int'>
looking for children: 86006362 <class 'int'>
looking for children: 86008082 <class 'int'>
looking for children: 86008078 <class 'int'>

Processing workflow run_id: 86007181
child_run_ids: [86005463, 86005462]
children_list: 8
sample children_list:
looking for children: 86003683 <class 'int'>
looking for children: 86005923 

In [15]:
import json

with open("automic_workflow_data.json", "w") as f:
    json.dump(save_data, f, indent=4)

print("Data saved to automic_workflow_data.json")

Data saved to automic_workflow_data.json


In [16]:
import sys
print("pyhton used in notebook:", sys.executable)
!{sys.executable} -m pip install pandas

pyhton used in notebook: C:\Users\nlr9894\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe
Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------- ----------------------------- 2.6/9.7 MB 18.9 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.7 MB 32.6 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 27.0 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ---------------------------------------- 12.3/12.3 MB 82.9 MB/s  0:00:00

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy